# 00 · Kiểm môi trường trước khi vẽ

Mục tiêu: biết **cái gì chạy được ngay** và **cái gì còn thiếu dữ liệu**, trước khi
mất thời gian với một notebook sẽ dừng ở ô thứ ba.

Nguyên tắc của cả bộ notebook: **không có số nào được bịa**. Ô nào thiếu dữ liệu
thì in ra thiếu gì và dừng, không vẽ hình rỗng.

In [ ]:
# Chạy từ thư mục gốc repo. Notebook CHỈ gọi tools/figures.py —
# không tự viết lại logic vẽ, để hình trong notebook và hình trong báo cáo
# luôn là cùng một hình.
import json, os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / 'tools' / 'figures.py').exists(), 'hãy chạy notebook từ thư mục gốc repo'
FIG = Path(os.environ.get('DIRE_FIG_OUT', 'figs'))

def fig_cmd(*args):
    """Chạy tools/figures.py, in lệnh rồi in đầu ra — lệnh in ra là lệnh tái lập."""
    cmd = [sys.executable, str(ROOT / 'tools' / 'figures.py'), *map(str, args)]
    print('$', ' '.join(cmd[1:]))
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
    r.check_returncode()
    return r

def sidecar(name):
    """Đọc sidecar: nguồn, n, nhãn. Không có sidecar = hình không kiểm được."""
    return json.loads((FIG / 'data' / f'{name}.json').read_text(encoding='utf-8'))

## 1. Repo có đủ thành phần không

In [ ]:
for p in ['tools/figures.py', 'src/dire_zero/schedule.py', 'docs/plans/research/figures-plan.md']:
    print(('OK  ' if (ROOT / p).exists() else 'MISS'), p)

## 2. Thư viện vẽ

`matplotlib` **không** phải phụ thuộc của `dire_zero` (package chỉ numpy/torch/PIL).
Notebook này cần nó, nên kiểm riêng.

In [ ]:
try:
    import matplotlib
    print('matplotlib', matplotlib.__version__)
except ImportError:
    print('THIẾU matplotlib — chạy:  pip install matplotlib')

In [ ]:
print(subprocess.run([sys.executable, str(ROOT / 'tools' / 'figures.py'), 'list'],
                       capture_output=True, text=True).stdout)

## 3. Dữ liệu đang ở đâu

Hai biến dưới đây là **đường dẫn trên máy bạn**, không phải hằng số của repo.
Đổi ở đây, không sửa trong code.

- `MAPS`: thư mục map đã giải nén, dạng `<split>/<generator>/*.npy`
  (giải từ `dire/test/lsun_bedroom/*.tar.gz` — map **của tác giả** ⇒ nhãn `R0`)
- `IMAGES`: thư mục ảnh đã `ingest`, dạng `<split>/<generator>/*.png`

In [ ]:
MAPS = Path(os.environ.get('DIRE_FIG_MAPS', 'data/maps-r0'))
IMAGES = Path(os.environ.get('DIRE_FIG_IMAGES', 'data/dire-zero-norm'))
MANIFEST = Path(os.environ.get('DIRE_FIG_MANIFEST', 'data/dire-zero-norm/manifest.tsv'))

for name, p in [('MAPS', MAPS), ('IMAGES', IMAGES), ('MANIFEST', MANIFEST)]:
    print(f'{name:9} {"có" if p.exists() else "CHƯA CÓ":7} {p}')

## 4. Kết luận: đợt nào chạy được

| Đợt | Hình | Cần gì | Chạy khi |
|---|---|---|---|
| 0 | F2, F3, A1 | chỉ repo | **luôn** — sang `01_hinh_schematic.ipynb` |
| 1 | F5, F1 | `MAPS` (+ `IMAGES` cho F1) | có map giải nén — sang `02_hinh_R0_tu_dire.ipynb` |
| 2 | A2, A3, A6 | `scores.npy`/`labels.npy`/manifest | **sau khi** có detector đã train |

Đợt 2 cần một lần chạy GPU thật (train + eval). Chưa có thì notebook đó sẽ
dừng ngay ở ô kiểm dữ liệu — đó là hành vi đúng, không phải lỗi.